# 🚀 ETL Silver → Gold

**Objetivo**: Popular o Star Schema (Gold Layer) a partir dos dados do Silver Layer

**Tabelas criadas**:
- `gold.dim_prdt` - Dimensão Produto
- `gold.dim_tmp` - Dimensão Tempo
- `gold.dim_desc` - Dimensão Desconto
- `gold.ft_vnd` - Fato Vendas

---

## 1️⃣ Imports e Configurações

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
from datetime import datetime
import numpy as np
from pathlib import Path

print("✅ Bibliotecas importadas com sucesso")

In [ ]:
# Database connection
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'amazon_sales',
    'user': 'postgres',
    'password': 'postgres'
}

# Files
SILVER_FILE = '../silver/data/amazon_products_cleaned_enhanced.csv'

print(f"📁 Arquivo Silver: {SILVER_FILE}")
print(f"🔌 Database: {DB_CONFIG['database']} @ {DB_CONFIG['host']}:{DB_CONFIG['port']}")

## 2️⃣ Carregar Dados do Silver Layer

In [ ]:
print("="*80)
print("📥 LOADING SILVER LAYER")
print("="*80)

df = pd.read_csv(SILVER_FILE)

print(f"\n✅ Loaded: {len(df):,} rows × {len(df.columns)} columns")
print(f"\n📊 Shape: {df.shape}")
print(f"\n🔍 Primeiras colunas: {list(df.columns[:10])}")

In [ ]:
# Verificar colunas essenciais
required_cols = ['asin', 'date', 'discount_bucket', 'has_active_coupon', 
                 'units_sold_last_month', 'price_tier', 'quality_score']

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"❌ Colunas faltando: {missing_cols}")
else:
    print("✅ Todas as colunas essenciais presentes")

df.head(3)

## 3️⃣ Conectar ao Banco de Dados

In [ ]:
def get_connection():
    """Cria conexão com o banco de dados"""
    return psycopg2.connect(**DB_CONFIG)

# Testar conexão
try:
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT version();")
    version = cur.fetchone()[0]
    print(f"✅ Conectado ao PostgreSQL")
    print(f"   Version: {version[:50]}...")
    conn.close()
except Exception as e:
    print(f"❌ Erro ao conectar: {e}")

## 4️⃣ Função: Popular Dimensão Tempo

In [ ]:
def populate_dim_tempo(df, conn):
    """Popula dim_tmp com todas as datas únicas"""
    print("\n📅 Populating dim_tmp...")
    
    dates = df['date'].dropna().unique()
    
    data = []
    for date_str in dates:
        dt = pd.to_datetime(date_str)
        data.append((
            dt.date(),
            dt.year,
            dt.month,
            dt.day,
            dt.dayofweek,
            dt.day_name(),
            (dt.month - 1) // 3 + 1,  # trimestre
            dt.isocalendar()[1],  # semana do ano
            dt.dayofweek >= 5,  # fim de semana
            dt.strftime('%Y-%m'),
            f"{dt.year}-Q{(dt.month-1)//3 + 1}"
        ))
    
    cur = conn.cursor()
    query = """
        INSERT INTO gold.dim_tmp 
        (data, ano, mes, dia, dia_semana, nome_dia_semana, trimestre, 
         semana_ano, eh_fim_semana, mes_ano, ano_trimestre)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (data) DO NOTHING
    """
    execute_batch(cur, query, data)
    conn.commit()
    print(f"   ✅ {len(data)} dates inserted")
    return len(data)

## 5️⃣ Função: Popular Dimensão Desconto

In [ ]:
def populate_dim_desconto(df, conn):
    """Popula dim_desc com combinações únicas"""
    print("\n🎁 Populating dim_desc...")
    
    discount_combos = df[['discount_bucket', 'has_active_coupon']].drop_duplicates()
    
    data = []
    for _, row in discount_combos.iterrows():
        bucket = row['discount_bucket']
        has_coupon = bool(row['has_active_coupon'])
        
        # Parse min/max do bucket
        if pd.isna(bucket) or bucket == 'No Discount':
            desc_min, desc_max = 0, 0
        elif bucket == '50%+':
            desc_min, desc_max = 50, 95
        else:
            parts = bucket.replace('%', '').split('-')
            desc_min = float(parts[0])
            desc_max = float(parts[1]) if len(parts) > 1 else desc_min
        
        data.append((str(bucket), has_coupon, desc_min, desc_max))
    
    cur = conn.cursor()
    query = """
        INSERT INTO gold.dim_desc 
        (faixa_desconto, tem_cupom, desconto_min, desconto_max)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (faixa_desconto, tem_cupom) DO NOTHING
    """
    execute_batch(cur, query, data)
    conn.commit()
    print(f"   ✅ {len(data)} discount combinations inserted")
    return len(data)

## 6️⃣ Função: Popular Dimensão Produto

In [ ]:
def populate_dim_produto(df, conn):
    """Popula dim_prdt"""
    print("\n📦 Populating dim_prdt...")
    
    produtos = df[['asin', 'title', 'brand', 'category', 'price_tier',
                   'best_seller_badge', 'sponsored_badge', 'is_promotable',
                   'available_for_purchase']].drop_duplicates('asin')
    
    data = []
    for _, row in produtos.iterrows():
        data.append((
            row['asin'],
            row['title'],
            row['brand'],
            row['category'],
            row['price_tier'],
            bool(row['best_seller_badge']),
            bool(row['sponsored_badge']),
            bool(row['is_promotable']),
            bool(row['available_for_purchase'])
        ))
    
    cur = conn.cursor()
    query = """
        INSERT INTO gold.dim_prdt 
        (asin, titulo, marca, categoria, faixa_preco, best_seller_badge,
         sponsored_badge, is_promotable, disponivel_compra)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (asin) DO UPDATE SET
            titulo = EXCLUDED.titulo,
            best_seller_badge = EXCLUDED.best_seller_badge,
            is_promotable = EXCLUDED.is_promotable,
            data_atualizacao = NOW()
    """
    execute_batch(cur, query, data)
    conn.commit()
    print(f"   ✅ {len(data)} products inserted/updated")
    return len(data)

## 7️⃣ Função: Popular Fato Vendas

In [ ]:
def populate_fato_vendas(df, conn):
    """Popula ft_vnd com join das FKs"""
    print("\n💰 Populating ft_vnd...")
    
    # Filtrar apenas registros com vendas
    df_facts = df[df['units_sold_last_month'].notna()].copy()
    print(f"   📊 {len(df_facts):,} records with sales data")
    
    cur = conn.cursor()
    
    # Lookup dims
    print("   🔍 Loading dimension keys...")
    cur.execute("SELECT prdt_key, asin FROM gold.dim_prdt")
    asin_to_key = dict(cur.fetchall())
    print(f"      ✅ dim_prdt: {len(asin_to_key):,} keys")
    
    cur.execute("SELECT tmp_key, data FROM gold.dim_tmp")
    date_to_key = {str(d): k for k, d in cur.fetchall()}
    print(f"      ✅ dim_tmp: {len(date_to_key):,} keys")
    
    cur.execute("SELECT desc_key, faixa_desconto, tem_cupom FROM gold.dim_desc")
    desconto_map = {(f, c): k for k, f, c in cur.fetchall()}
    print(f"      ✅ dim_desc: {len(desconto_map):,} keys")
    
    # Build fact records
    print("   🔨 Building fact records...")
    data = []
    skipped = 0
    for _, row in df_facts.iterrows():
        prdt_key = asin_to_key.get(row['asin'])
        tmp_key = date_to_key.get(str(row['date']))
        desc_key = desconto_map.get((str(row['discount_bucket']), bool(row['has_active_coupon'])))
        
        if not (prdt_key and tmp_key and desc_key):
            skipped += 1
            continue
        
        data.append((
            prdt_key,
            tmp_key,
            desc_key,
            int(row['units_sold_last_month']) if pd.notna(row['units_sold_last_month']) else 0,
            float(row['revenue_last_month']) if pd.notna(row['revenue_last_month']) else 0,
            float(row['final_price']) if pd.notna(row['final_price']) else 0,
            float(row['rating']) if pd.notna(row['rating']) else None,
            int(row['review_count']) if pd.notna(row['review_count']) else 0,
            float(row['quality_score']) if pd.notna(row['quality_score']) else None,
            float(row['discount_pct']) if pd.notna(row['discount_pct']) else 0,
            int(row['hour']) if pd.notna(row['hour']) else 0,
            pd.to_datetime(row['collected_at']),
            str(row['price_imputation_tier'])
        ))
    
    print(f"   💾 Inserting {len(data):,} records...")
    query = """
        INSERT INTO gold.ft_vnd 
        (prdt_key, tmp_key, desc_key, unidades_vendidas, receita_estimada,
         preco_final, rating, total_reviews, quality_score, percentual_desconto,
         hora_coleta, data_coleta, origem_preco)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    execute_batch(cur, query, data, page_size=1000)
    conn.commit()
    print(f"   ✅ {len(data):,} fact records inserted")
    if skipped > 0:
        print(f"   ⚠️  {skipped:,} records skipped (missing FK references)")
    
    return len(data), skipped

## 8️⃣ Executar ETL Completo

In [ ]:
print("="*80)
print("🚀 ETL SILVER → GOLD")
print("="*80)

# Connect DB
print("\n🔌 Connecting to database...")
conn = get_connection()
print("   ✅ Connected")

try:
    # Populate dimensions (order matters!)
    tempo_count = populate_dim_tempo(df, conn)
    desc_count = populate_dim_desconto(df, conn)
    prdt_count = populate_dim_produto(df, conn)
    
    # Populate fact
    fact_count, skipped_count = populate_fato_vendas(df, conn)
    
    print("\n" + "="*80)
    print("✅ ETL COMPLETED SUCCESSFULLY")
    print("="*80)
    
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    conn.rollback()
    raise
finally:
    conn.close()
    print("\n🔌 Database connection closed")

## 9️⃣ Validação e Estatísticas

In [ ]:
print("="*80)
print("🔍 VALIDATION & STATISTICS")
print("="*80)

conn = get_connection()
cur = conn.cursor()

# Count tables
print("\n📊 Table Counts:")
print("-" * 60)

cur.execute("SELECT COUNT(*) FROM gold.dim_prdt")
prdt_count = cur.fetchone()[0]
print(f"📦 dim_prdt: {prdt_count:,} rows")

cur.execute("SELECT COUNT(*) FROM gold.dim_tmp")
tmp_count = cur.fetchone()[0]
print(f"📅 dim_tmp: {tmp_count:,} rows")

cur.execute("SELECT COUNT(*) FROM gold.dim_desc")
desc_count = cur.fetchone()[0]
print(f"🎁 dim_desc: {desc_count:,} rows")

cur.execute("SELECT COUNT(*) FROM gold.ft_vnd")
vnd_count = cur.fetchone()[0]
print(f"💰 ft_vnd: {vnd_count:,} rows")

# Business metrics
print("\n💵 Business Metrics:")
print("-" * 60)

cur.execute("SELECT SUM(receita_estimada) FROM gold.ft_vnd")
total_revenue = cur.fetchone()[0]
print(f"Total Revenue: ${total_revenue:,.2f}")

cur.execute("SELECT SUM(unidades_vendidas) FROM gold.ft_vnd")
total_units = cur.fetchone()[0]
print(f"Total Units Sold: {total_units:,}")

cur.execute("SELECT AVG(rating) FROM gold.ft_vnd WHERE rating IS NOT NULL")
avg_rating = cur.fetchone()[0]
print(f"Average Rating: {avg_rating:.2f}")

cur.execute("SELECT COUNT(*) FROM gold.dim_prdt WHERE is_promotable = TRUE")
promotable = cur.fetchone()[0]
print(f"Promotable Products: {promotable:,} ({promotable/prdt_count*100:.1f}%)")

conn.close()

print("\n" + "="*80)
print("🎉 GOLD LAYER READY FOR POWER BI!")
print("="*80)

## 🔟 Consultas de Teste

In [ ]:
# Top 10 produtos por receita
query = """
SELECT 
    p.titulo,
    p.marca,
    p.categoria,
    f.unidades_vendidas,
    f.receita_estimada,
    f.rating
FROM gold.ft_vnd f
JOIN gold.dim_prdt p ON f.prdt_key = p.prdt_key
ORDER BY f.receita_estimada DESC
LIMIT 10;
"""

conn = get_connection()
df_top10 = pd.read_sql_query(query, conn)
conn.close()

print("🏆 TOP 10 PRODUTOS POR RECEITA:\n")
df_top10

In [ ]:
# Resumo por categoria
query = "SELECT * FROM gold.vw_resumo_categoria ORDER BY receita_total DESC;"

conn = get_connection()
df_cat = pd.read_sql_query(query, conn)
conn.close()

print("📊 RESUMO POR CATEGORIA:\n")
df_cat

In [ ]:
# Efetividade de descontos
query = """
SELECT 
    faixa_desconto,
    COUNT(*) as total_categorias,
    SUM(qtd_produtos) as total_produtos,
    AVG(vendas_medias) as media_vendas,
    SUM(receita_total) as receita_total
FROM gold.vw_efetividade_desconto
GROUP BY faixa_desconto
ORDER BY receita_total DESC;
"""

conn = get_connection()
df_desc = pd.read_sql_query(query, conn)
conn.close()

print("💸 EFETIVIDADE DE DESCONTOS:\n")
df_desc

---

## ✅ Checklist Final

Antes de conectar ao Power BI, verifique:

- [ ] `dim_prdt` tem registros (produtos únicos)
- [ ] `dim_tmp` tem registros (datas únicas)
- [ ] `dim_desc` tem registros (combinações desconto/cupom)
- [ ] `ft_vnd` tem ~30K+ registros
- [ ] Total revenue é ~$47M
- [ ] Views estão funcionando

**Próximo passo**: Conectar Power BI ao schema `gold` e criar dashboards! 🚀